In [12]:
import pandas as pd
import numpy as np
import os
import sys

print("=" * 60) 
print("Step 1: Dataset Creation & Cleaning")
print("Works with ANY network traffic CSV dataset")
print("=" * 60)

DATA_DIR    = "data"
OUTPUT_PATH = os.path.join(DATA_DIR, "final_dataset.csv")

if not os.path.exists(DATA_DIR):
    print(f"ERROR: '{DATA_DIR}/' folder not found.")
    print("Create a 'data/' folder and place your CSV files inside.")
    print("Supported datasets: CICIDS 2017, NSL-KDD, UNSW-NB15, CIC-DDoS, custom")
    sys.exit(1)

files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])
if not files:
    print(f"ERROR: No CSV files in '{DATA_DIR}/'.")
    sys.exit(1)

# ── Read all CSVs ─────────────────────────────────────────────────────────────
df_list = []
for file in files:
    path = os.path.join(DATA_DIR, file)
    try:
        print(f"Reading {file}...")
        df = pd.read_csv(path, low_memory=False, on_bad_lines="skip")
        df.columns = df.columns.str.strip()
        df = df.loc[:, ~df.columns.str.lower().str.startswith("unnamed")]
        print(f"  -> {len(df):,} rows, {len(df.columns)} columns")
        df_list.append(df)
    except Exception as e:
        print(f"  WARNING: {file}: {e}")

if not df_list:
    print("ERROR: No files loaded.")
    sys.exit(1)

# ── Align columns across files ────────────────────────────────────────────────
common_cols = set(df_list[0].columns)
for df in df_list[1:]:
    common_cols &= set(df.columns)
df_list  = [df[sorted(common_cols)] for df in df_list]
dataset  = pd.concat(df_list, ignore_index=True)
print(f"\nTotal rows: {len(dataset):,} | Columns: {len(dataset.columns)}")

# ── Auto-detect label column ──────────────────────────────────────────────────
# Supports CICIDS 2017, NSL-KDD, UNSW-NB15, CIC-DDoS, and custom datasets
LABEL_CANDIDATES = [
    "Label", "label", "LABEL",
    "Category", "category",
    "Class", "class",
    "attack_cat", "Attack",
    "type", "Type",
    "target", "Target",
]
label_col = next((c for c in LABEL_CANDIDATES if c in dataset.columns), None)

if label_col is None:
    print("\nERROR: Could not find a label column.")
    print("Columns found:", list(dataset.columns))
    print("Rename your label column to 'Label' and re-run.")
    sys.exit(1)

if label_col != "Label":
    print(f"Found label column: '{label_col}' → renaming to 'Label'")
    dataset = dataset.rename(columns={label_col: "Label"})

# ── Convert label to binary (works for ANY dataset) ───────────────────────────
# Handles: string labels, numeric labels, multi-class labels
# Any non-BENIGN / non-Normal / non-0 label = ATTACK (1)
print(f"\nUnique labels found: {list(dataset['Label'].dropna().unique()[:15])}")

BENIGN_LABELS = {"benign", "normal", "legitimate", "0", "none", "-", "needmanuallabel"}

if pd.api.types.is_numeric_dtype(dataset["Label"]):
    # Numeric: 0 = benign, anything else = attack
    dataset["Label"] = (dataset["Label"] != 0).astype(int)
else:
    dataset["Label"] = dataset["Label"].apply(
        lambda x: 0 if str(x).strip().lower() in BENIGN_LABELS else 1
    )

print(f"\nLabel distribution:")
print(f"  BENIGN (0): {(dataset['Label']==0).sum():,}")
print(f"  ATTACK (1): {(dataset['Label']==1).sum():,}")

attack_pct = (dataset['Label']==1).mean()*100
if attack_pct < 1:
    print("WARNING: Less than 1% attack samples — dataset may be heavily imbalanced")
if attack_pct > 99:
    print("WARNING: More than 99% attack samples — check your label column")

# ── Fix inf / NaN ─────────────────────────────────────────────────────────────
numeric_cols = dataset.select_dtypes(include="number").columns
inf_count    = dataset[numeric_cols].isin([float("inf"), float("-inf")]).sum().sum()
if inf_count > 0:
    print(f"\nReplacing {inf_count:,} inf values with NaN...")
    dataset[numeric_cols] = dataset[numeric_cols].replace(
        [float("inf"), float("-inf")], np.nan
    )

nan_count = dataset[numeric_cols].isnull().sum().sum()
if nan_count > 0:
    print(f"Imputing {nan_count:,} NaN values with column medians...")
    dataset[numeric_cols] = dataset[numeric_cols].fillna(
        dataset[numeric_cols].median()
    )

# ── Drop non-numeric columns (IP, MAC, timestamps) ───────────────────────────
non_numeric = [c for c in dataset.select_dtypes(exclude="number").columns
               if c != "Label"]
if non_numeric:
    print(f"Dropping non-numeric columns: {non_numeric}")
    dataset = dataset.drop(columns=non_numeric)

# ── Balanced stratified sampling ──────────────────────────────────────────────
# 75k per class = 150k total — fits RAM on any machine
# If a class has fewer than 75k, uses all available samples
SAMPLE_PER_CLASS = 75_000
benign_df = dataset[dataset["Label"] == 0]
attack_df = dataset[dataset["Label"] == 1]

n_b = min(SAMPLE_PER_CLASS, len(benign_df))
n_a = min(SAMPLE_PER_CLASS, len(attack_df))

if n_a < 100:
    print(f"ERROR: Only {n_a} attack samples found — too few to train on.")
    sys.exit(1)

final = pd.concat([
    benign_df.sample(n=n_b, random_state=42),
    attack_df.sample(n=n_a, random_state=42)
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nFinal balanced dataset:")
print(f"  BENIGN (0): {(final['Label']==0).sum():,}")
print(f"  ATTACK (1): {(final['Label']==1).sum():,}")
print(f"  Total     : {len(final):,}")

final.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved: {OUTPUT_PATH}")
print("create_dataset.py completed successfully!")

Step 1: Dataset Creation & Cleaning
Works with ANY network traffic CSV dataset
Reading log5.pcap_Flow.csv...
  -> 330 rows, 79 columns

Total rows: 330 | Columns: 79

Unique labels found: ['NeedManualLabel']

Label distribution:
  BENIGN (0): 330
  ATTACK (1): 0
ERROR: Only 0 attack samples found — too few to train on.


SystemExit: 1

C:\Users\Dell\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
